<a href="https://colab.research.google.com/github/hussain-azmat/colab_work/blob/main/Pneumonia_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Configuration



Dataset available on kaggle : https://www.kaggle.com/paultimothymooney/chest-xray-pneumonia

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm.notebook import tqdm

In [ ]:
class CFG:

    epochs = 20                                         # No. of epochs of training the model
    lr = 0.001                                         # Learning rate
    batch_size = 16                                    # Batch Size For Dataset

    model_name = 'tf_efficientnet_b4_ns'               # Model name (We are going to import model from timm)
    img_size = 224

    # Going to be use for loading dataset
    DATA_DIR = "/contents/drive/MyDrive/datasets/chest_xray_data"                       # Data Directory
    TEST = 'test'                                      # Test folder name in data directory
    TRAIN = 'train'                                    # Train folder name in data directory
    VAL ='val'                                         # Valid folder name in data directory

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("On which device we are on : {}".format(device))

On which device we are on : cpu


# Image Transformation and Load Dataset

In [ ]:
from torchvision import transforms as T, datasets
#from helper import show_image

In [ ]:
train_transform =  T.Compose([

    T.Resize(size = (CFG.img_size, CFG.img_size)),
    T.RandomRotation(degrees = (-20, +20)),
    T.ToTensor(), #(h,w,c) -> (c,h,w)
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

valid_transform =  T.Compose([

    T.Resize(size = (CFG.img_size, CFG.img_size)),
    T.ToTensor(), #(h,w,c) -> (c,h,w)
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform =  T.Compose([

    T.Resize(size = (CFG.img_size, CFG.img_size)),
    T.ToTensor(), #(h,w,c) -> (c,h,w)
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
train_path = os.path.join(CFG.DATA_DIR, CFG.TRAIN)
valid_path = os.path.join(CFG.DATA_DIR, CFG.VAL)
test_path = os.path.join(CFG.DATA_DIR, CFG.TEST)

trainset = datasets.ImageFolder(train_path, transform = train_transform)
validset = datasets.ImageFolder(valid_path, transform = valid_transform)
testset = datasets.ImageFolder(test_path, transform = test_transform)

FileNotFoundError: [Errno 2] No such file or directory: 'chest_xray_data.zip/train'

In [ ]:
print("Trainset Size : {}".format(len(trainset)))
print("Validset Size : {}".format(len(validset)))
print("Testset Size : {}".format(len(testset)))

In [ ]:
image, label = trainset[2]

class_name = ['NORMAL', 'PNEUMONIA']

show_image(image, class_name[label])

# Load Dataset into Batches

In [ ]:
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from helper import show_grid

In [ ]:
trainloader = DataLoader(trainset, batch_size = CFG.batch_size, shuffle = True)
validloader = DataLoader(validset, batch_size = CFG.batch_size, shuffle = True)
testloader = DataLoader(testset, batch_size = CFG.batch_size, shuffle = True)

In [ ]:
print("No. of batches in trainloader : {}".format(len(trainloader)))
print("No. of Total examples : {}".format(len(trainloader.dataset)))

print(len(trainset))

In [ ]:
dataiter =  iter(trainloader)
images, labels =  next(dataiter)

out = make_grid(images, nrow = 4)

show_grid(out, title = [class_name[x] for x in labels])

# Fine Tuning EfficientNet Model

In [ ]:
from torch import nn
import torch.nn.functional as F
import timm

model = timm.create_model(CFG.model_name, pretrained = True)

for param in model.parameters():
    param.required_grad = False

model.classifier = nn.Sequential(

    nn.Linear(in_features = 1792, out_features = 625),
    nn.ReLU(),
    nn.Dropout(p = 0.3),
    nn.Linear(in_features = 625, out_features = 256),
    nn.ReLU(),
    nn.Linear(in_features = 256, out_features = 2)
)

In [ ]:
from torchsummary import summary

summary(model, input_size = (3, 224, 224))

# Build a Simple Trainer

In [ ]:
from helper import accuracy
from tqdm import tqdm

In [ ]:
class PneumoniaTrainer():

    def __init__(self,criterion = None,optimizer = None,schedular = None):
        self.criterion = criterion
        self.optimizer = optimizer
        self.schedular = schedular

    def train_batch_loop(self,model,trainloader):

        train_loss = 0.0
        train_acc = 0.0

        for images,labels in tqdm(trainloader):

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = self.criterion(logits,labels)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            train_loss += loss.item()
            train_acc += accuracy(logits,labels)

        return train_loss / len(trainloader), train_acc / len(trainloader)


    def valid_batch_loop(self, model, validloader):
        valid_loss = 0.0
        valid_acc = 0.0

        for images,labels in tqdm(validloader):

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = self.criterion(logits,labels)

            valid_loss += loss.item()
            valid_acc += accuracy(logits,labels)

        return valid_loss / len(validloader), valid_acc / len(validloader)



    def fit(self,model,trainloader,validloader,epochs):

        model = model.to(device)
        valid_min_loss = np.Inf

        for i in range(epochs):

            model.train()
            avg_train_loss, avg_train_acc = self.train_batch_loop(model,trainloader)

            model.eval()
            avg_valid_loss, avg_valid_acc = self.valid_batch_loop(model,validloader)

            if avg_valid_loss <= valid_min_loss :
                print("Valid_loss decreased {} --> {}".format(valid_min_loss,avg_valid_loss))
                torch.save(model.state_dict(),'PneumoniaModel.pt')
                valid_min_loss = avg_valid_loss


            print("Epoch : {} Train Loss : {:.6f} Train Acc : {:.6f}".format(i+1, avg_train_loss, avg_train_acc))
            print("Epoch : {} Valid Loss : {:.6f} Valid Acc : {:.6f}".format(i+1, avg_valid_loss, avg_valid_acc))

# Training Model


Trained on google colab : https://colab.research.google.com/drive/1C5nNPj7OLYMGnNvWBU5W2zSXfgWDdXCo?usp=sharing

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = CFG.lr)
Scheduler = None
trainer = PneumoniaTrainer(criterion,optimizer, Scheduler)
trainer.fit(model,trainloader,validloader,epochs = CFG.epochs)

# Plot Results

In [ ]:
from helper import view_classify

model.load_state_dict(torch.load('/content/ColabPneumoniaModel.pt'))
model.eval()

avg_test_loss, avg_test_acc = trainer.valid_batch_loop(model,testloader)


print("Test Loss : {}".format(avg_test_acc))
print("Test Acc : {}".format(avg_test_loss))

In [ ]:
import torch.nn.functional as F

image,label = testset[3]

ps = model(image.to(device).unsqueeze(0))
ps = F.softmax(ps,dim = 1)

view_classify(image,ps,label)